# What the antenna buys: 21-cm retention under an identical filter

One sidereal day of the same GSM16 sky behind the same canyon horizon at the
same site, simulated three times with three different beams. Each antenna's
foregrounds get their own uncentered SVD, and each antenna's 21-cm ensemble is
filtered in that antenna's own basis, so every quantity in a panel belongs to
the antenna named in it.

Both curves in every panel are **antenna temperature**, ground pickup
included. The 21-cm models are multiplied by that antenna's beam-weighted
open-sky fraction before filtering, because an isotropic signal reaches the
antenna temperature attenuated by it. They are not sky-referred amplitudes.

**Why three panels rather than three curves.** Each antenna has a different
foreground residual, a different open-sky fraction and a different eigenbasis.
On one panel the reader would inevitably compare one antenna's foregrounds
against another's ensemble. The grey bands are *not* the same band drawn three
times.

**Why a band rather than every model.** Both were rendered. The ensemble's
extremes run a factor of 2--2.5 beyond the 5--95 band, so little is hidden, but
1769 individual curves have no crisp edge and the crossing -- how far right the
black curve travels before it drops under the grey -- is what the figure is
for. The dashed median is also lost in the haze. The median is an ensemble
statistic and not a single model: it is a different model at almost every $N$,
though a real model tracks it to within 1.3x across the axis.

**The isotropic beam** is the chromaticity-free reference. It still sees the
real horizon, so its open-sky fraction is pure geometry and achromatic; what
little structure survives in its panel is the sky's, not an antenna's. Its
residual leaves the bottom of the frame near $N = 6$, where it reaches machine
precision.

**The Vivaldi feed** is the HERA Phase II design used in isolation without its
dish, which is the antenna EIGSEP's first suspension flew in October 2024. It
couples considerably better to the sky than the bowtie -- it starts with more
signal -- and still retains less of it at matched foreground suppression. This
is not a statement about the feed in the configuration it was designed for.

**Scope.** Zenith pointing, nominal position, GSM16, no noise and no receiver
systematics. Read this as a statement about spectral overlap, not a sensitivity
forecast.

Data: `beam_comparison.npz`. Full derivation from the raw simulation output:
`mock_analysis/horizon_position/notebooks/beam_comparison.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
d = np.load("beam_comparison.npz", allow_pickle=True)
n_modes = d["n_modes"]                 # foreground modes filtered (x-axis)
order = [str(x) for x in d["order"]]   # panel order, left to right
labels = {b: str(l) for b, l in zip(order, d["labels"])}
# Per antenna: foreground residual [K] and the 5/50/95 percentiles of the
# retained 21-cm RMS [K], both already in antenna temperature.
fg_resid = {b: d["fg_resid"][i] for i, b in enumerate(order)}
t21_pct = {b: d["t21_pct"][i] for i, b in enumerate(order)}
C_21 = "0.40"
print(f"{len(order)} antennas, {n_modes[-1]} modes on the axis")

In [ ]:
def make_figure(n_modes, fg_resid, t21_pct, order, labels, c_21, path):
    """Three panels of the Fig. 1 axes, one per antenna.

    Shared y so the panels can be read against each other; the label sits
    inside each panel rather than as a title, to keep the row compact enough
    for a two-column `figure*`.
    """
    fig, axs = plt.subplots(1, 3, figsize=(7.0, 2.6), layout="constrained",
                            sharey=True)
    for ax, b in zip(axs, order):
        ax.fill_between(n_modes, t21_pct[b][0], t21_pct[b][2], color=c_21,
                        alpha=0.25, lw=0, zorder=0)
        ax.plot(n_modes, t21_pct[b][1], color=c_21, lw=1.3, ls="--", zorder=1)
        ax.plot(n_modes, fg_resid[b], color="k", lw=1.8, zorder=2)
        ax.set_yscale("log")
        ax.set_xlim(0, n_modes[-1])
        ax.set_ylim(1e-5, 3e3)
        ax.grid(True, which="both", ls=":", lw=0.5, alpha=0.5)
        ax.tick_params(labelsize=7)
        ax.set_xlabel("Foreground modes filtered", fontsize=8)
        ax.text(0.045, 0.94, labels[b], transform=ax.transAxes, va="top",
                fontsize=7.5)
    axs[0].set_ylabel("Residual RMS [K]", fontsize=8)
    # Centre right of the first panel: the white band between the 21 cm
    # ensemble and the foreground curve, which the isotropic residual clears
    # by N ~ 4. Lower right put the box on that residual's descent; upper
    # right collides with the panel label.
    axs[0].legend(
        handles=[
            Line2D([], [], color="k", lw=1.8, label="Beam-weighted foregrounds"),
            Line2D([], [], color=c_21, lw=1.3, ls="--", label="21-cm models"),
        ],
        fontsize=6, loc="center right", framealpha=0.92,
    )
    fig.savefig(path, bbox_inches="tight", dpi=600)
    return fig

In [ ]:
fig = make_figure(n_modes, fg_resid, t21_pct, order, labels, C_21,
                  "beam_comparison.pdf")